### Day 11 Assignment: Power BI, OpenSharing, Lakehouse Federation & Lakebase

### Basic Tasks

#### 1. Connect Power BI Desktop to a Databricks SQL warehouse and build one report against a Gold table

In [0]:
SELECT *
FROM dev.gold.sales_summary
LIMIT 10;

In [0]:
SELECT
    COUNT(*) AS total_rows,
    ROUND(SUM(sale_amount),2) AS total_sales,
    SUM(quantity) AS total_quantity
FROM dev.gold.sales_summary;

![image_1789370775262.png](./image_1789370775262.png "image_1789370775262.png")

#### 2. Create a Unity Catalog connection to an external PostgreSQL database and a foreign catalog exposing one of its tables. 



Created a Unity Catalog connection to an external **PostgreSQL database hosted on Neon** and created a **foreign catalog** named `postgres_conn_catalog`.

The external PostgreSQL database contains a `customers` table under the `public` schema. Databricks exposes this table as a **Foreign** table, allowing it to be queried directly without copying the data into a Delta table.

**Architecture:**

Neon PostgreSQL → Unity Catalog Connection → Foreign Catalog → `public.customers`

The federated table was validated using:

```sql
SELECT *
FROM postgres_conn_catalog.public.customers;

In [0]:
SELECT *
FROM postgres_conn_catalog.public.customers

#### 3. Lakebase

I would choose **Lakebase instead of a Delta table** when building an application that requires frequent, low-latency reads and writes.

Lakebase is suited for operational and transactional application workloads where data is continuously updated and applications need quick access to the latest state. Delta tables are generally better suited for large-scale analytical workloads such as data engineering, BI, and reporting.

**In short:**

Lakebase → Application / transactional workloads → Fast reads and writes

Delta Lake → Analytical workloads → Large-scale data processing and BI

### Intermediate Tasks 



#### 4. Power BI Service and Data Freshness

The Power BI report was created in **Power BI Desktop** using the Databricks SQL Warehouse connection. The connection is configured in **Import mode**, meaning data from the Databricks Gold table is imported into the Power BI semantic model.

Publishing to Power BI Service normally requires signing in with a **work or school Microsoft account**. Since a work account is not available for this exercise, the publishing step could not be completed practically. The expected workflow is:

**Power BI Desktop → Publish → Power BI Service → Workspace → Semantic Model + Report**

#### Data Freshness: Import vs. Live Query

**1. Import mode — Scheduled Refresh**

With Import mode, a copy of the data is stored in the Power BI semantic model. The report does not automatically query Databricks every time it is opened.

Therefore, the data can become **stale between refreshes**.

For example, if the Databricks Gold table is updated at 10:00 AM but the Power BI semantic model is scheduled to refresh every 6 hours, the report may continue showing the older data until the next successful refresh.

A scheduled refresh can also result in stale data if the refresh fails or is delayed.

**2. DirectQuery / Live Query**

With DirectQuery, the data is not imported into the Power BI semantic model. Instead, queries are sent to the underlying Databricks SQL Warehouse when report visuals need data.

Therefore, DirectQuery generally provides **fresher data** than Import mode because the report queries the source rather than relying on a stored copy.

However, report/dashboard caching, source connectivity issues, query delays, or other Power BI caching behavior can still affect when the latest data is displayed.

#### Conclusion

**Import → Data stored in Power BI → Requires scheduled/on-demand refresh → Can become stale**

**DirectQuery → Queries Databricks at report/query time → Less dependent on scheduled refresh → Generally fresher data**

#### 5. Lakehouse Federation — Federated Join

Performed a federated join between a **native Unity Catalog Delta table** and an **external PostgreSQL table**.

The native Gold table is:

`dev.gold.sales_summary`

The PostgreSQL table exposed through Unity Catalog is:

`postgres_conn_catalog.public.customers`

#### Confirming No Physical Copy

The PostgreSQL table is registered as a **Foreign** table with **Data Source = PostgreSQL**, confirming that the data remains in the external database and is queried through federation rather than copied into Databricks.

In [0]:
SELECT c.customer_id, c.customer_name, c.city, s.region
FROM dev.gold.sales_summary s
JOIN postgres_conn_catalog.public.customers c
  ON s.customer_id = c.customer_id
LIMIT 10;


#### 6. Databricks-to-Databricks OpenSharing

Created a Databricks-to-Databricks **OpenSharing** share for the Gold table:

`dev.gold.sales_summary`

A recipient representing the second Databricks workspace was created using the workspace's **Sharing Identifier**, and access to the share was granted to that recipient.

This demonstrates cross-workspace data sharing through **Delta Sharing/OpenSharing**, without requiring the recipient to receive a physical copy of the source table.

#### Supported Content Types

OpenSharing supports several data and AI asset types, including:

- Tables
- Views
- Volumes
- Notebooks
- Materialized views
- Streaming tables
- Models and other supported assets

For this exercise, the shared asset is the Gold **table** `dev.gold.sales_summary`.

![image_1789408084526.png](./image_1789408084526.png "image_1789408084526.png")

![image_1789408099622.png](./image_1789408099622.png "image_1789408099622.png")

![image_1789453568627.png](./image_1789453568627.png "image_1789453568627.png")


####  7. Import an existing Power BI file into an AI/BI dashboard
Imported the existing Power BI report into a **Databricks AI/BI Dashboard** using the Power BI template (`.pbit`) file.

The migration process analyzed the Power BI report and recreated the dashboard, including the underlying dataset, metric view, measures, dimensions, and visualizations.

The imported dashboard successfully created:

- **1 Metric View**
- **1 Dashboard Page**
- **3 Dashboard Widgets**
- **3 Measures**
- **14 Dimensions**
- **1 Dataset**

#### What Translated Cleanly

The following components were successfully migrated:

- **Total Sale Amount** was recreated as a KPI/card showing the same overall sales value.
- **Sale Amount by Region** was successfully recreated as a regional column chart.
- The dashboard page and the three main visualizations were successfully created.
- The underlying fields, measures, and dimensions were available in the migrated dashboard.
- The migrated dashboard was verified successfully and connected to live data.

#### What Did Not Translate Exactly

The **Sale Amount Over Time** visualization was recreated, but its visualization behavior/format was not identical to the original Power BI line chart. The migrated chart uses `sale_date` and displays the data differently rather than preserving the exact original Power BI line-chart configuration.

Some visual formatting and layout details may also require manual adjustment after migration.

#### Conclusion

The Power BI-to-AI/BI migration successfully transferred the main analytical content and data model, but visualization formatting and some chart configurations may require manual refinement after migration.

This demonstrates that an existing Power BI report can be migrated to a Databricks AI/BI Dashboard while preserving the core data, measures, dimensions, and major visualizations.

![image_1789454729221.png](./image_1789454729221.png "image_1789454729221.png")

###  Advanced Tasks 


#### 8. OpenSharing Decision Matrix

The choice between Databricks-to-Databricks and Databricks-to-Open depends mainly on whether the partner has their own Databricks workspace and what type of assets they need.

If the partner **has their own Databricks workspace and needs tables**, use **Databricks-to-Databricks OpenSharing**. This provides native Databricks integration and allows the recipient to access shared data through their Databricks environment.

If the partner **has their own Databricks workspace and needs AI assets**, use **Databricks-to-Databricks OpenSharing**. This is preferred because Databricks-native assets such as supported models, notebooks, volumes, and metric views can be shared through the Databricks-to-Databricks workflow.

If the partner **does not have a Databricks workspace and needs tables**, use **Databricks-to-Open OpenSharing**. This allows external recipients on non-Databricks platforms to consume shared data without requiring their own Databricks workspace.

If the partner **does not have a Databricks workspace and needs AI assets**, Databricks-to-Open should only be used if the specific asset is supported for external sharing. Many Databricks-native AI assets require Databricks-to-Databricks sharing, so the supported asset type must be checked before choosing this option.

In summary:

Partner has Databricks → Databricks-to-Databricks

Partner does not have Databricks → Databricks-to-Open

For Databricks-native AI assets, Databricks-to-Databricks is generally the preferred option.

#### 9. Query Federation vs. Nightly Copy Pipeline

Query Federation and a nightly copy pipeline have different trade-offs in terms of data freshness, query performance, and workload on the source PostgreSQL database.

I would prefer **Lakehouse Federation** when the business requires fresh or near-real-time data and the analytical query volume is manageable. Federation allows Databricks to query the PostgreSQL source directly without maintaining a separate copy.

Federation starts to make less sense under the following conditions:

- The business can **tolerate a few hours of data delay**, making a nightly batch copy acceptable.
- The source data is naturally **batch-oriented** and does not require real-time access.
- There is a **high volume of analytical queries** or complex queries against the PostgreSQL source.
- Repeated federated queries create additional **compute workload and query latency** on PostgreSQL and could impact the performance of operational/OLTP workloads.
- A Delta copy would provide **faster and more scalable analytical query execution** because the workload can be handled by the lakehouse rather than repeatedly querying the operational database.

Therefore:

**Freshness is critical + manageable query volume → Federation**

**Few hours of latency is acceptable + high analytical query volume → Nightly Delta Copy**

The final decision should balance the required data freshness against source-system load, query performance, and operational cost.


#### 10. LTAP Architecture — Real-Time Inventory Application

For a real-time inventory-check application, I would use **Lakebase for OLTP workloads** and the **Lakehouse/Delta tables for OLAP workloads**.

The application would write inventory changes such as stock updates, orders, and inventory adjustments to **Lakebase**. Lakebase would serve as the operational system because the application requires frequent transactional writes and low-latency reads of the latest inventory state.

The operational data from Lakebase would then be **synchronized into the Lakehouse** using an appropriate data pipeline. The Lakehouse would maintain Delta tables containing the synchronized inventory data and historical information required for analytics.

The responsibilities would be separated as follows:

**Lakebase / OLTP**
- Owned operationally by the application/engineering team.
- Handles real-time inventory reads and transactional writes.
- Maintains the latest operational inventory state.

**Lakehouse / OLAP**
- Data Engineers own the ingestion/synchronization pipelines and data platform.
- Delta tables provide the analytical layer for historical and large-scale analysis.
- Data Analysts and BI users consume the curated Delta data for reporting and analytics.

**Data flow:**

Inventory Application → Lakebase → Synchronization Pipeline → Delta/Lakehouse → BI & Analytics

This architecture keeps transactional application workloads separate from analytical workloads while allowing the same business data to support both real-time operations and historical analytics.